# Extraction of spectra from the UAV images

**Original file:** `1_Spectors/1_JypiterLabProcessing/1_Correction_extraction_FullData.ipynb` in the author's project archive.

Selects the flight lines of each acquisition date, adjusts them to the plot boundaries and extracts the spectra of each plot. The output tables feed notebooks 02 and 03, and the last section prepares the vector files shared in the published dataset.

**Status:** kept for reference; it does not run as it is. It needs Python 3.6, library versions of that time, files that are not shared and, in places, manual changes between runs. `<PROJECT_ROOT>`, `<RAW_DATA>` and `<CONDA_ENVS>` replace local folders. To reproduce the results, use the scripts in the main folder of this repository.

## 1. Setup

### Packages

Libraries used for reading rasters and vector files. *(Original cell 1.)*

In [ ]:
from affine import Affine
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
import copy
from descartes import PolygonPatch
import gdal
import gdalconst
import fiona
import geopandas as gpd
import glob
import itertools
import matplotlib
import matplotlib.pyplot as plt
%matplotlib qt
import numpy as np
import os
import osr
import pandas as pd
import progressbar as pbar
import pylab
import rasterio
import rasterio.plot
import rasterio.mask
from shapely.geometry import LinearRing as LinearR
from shapely.geometry import Point
from shapely.geometry import LineString as LineS
from shapely.geometry.polygon import Polygon as PolyS
import spectral.io.envi as envi
import sys

HomeDir = r'<PROJECT_ROOT>\DataProcessing\1_Script_source'
os.chdir(HomeDir)
import VIs_formulation_update20190402 as VIf

print("This notebook was built with:")
print("python version: %s " % sys.version)

## 2. Selection of flight lines

### Input files

Selects the acquisition (`yearRef`, `acqRef`, changed manually for each date) and lists its hyperspectral images. *(Original cell 2.)*

In [ ]:
yearRef = 2014
acqRef  = 1

# select the type of extraction - full data or after buffer
dataTypeSel = 'Buffer'

if( yearRef == 2017 ):
    if( acqRef == 1 ):
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170509\HYMSY'  # Hyper - Acq 1
        os.chdir( HomeDir )
        PatternUpload = '.\\' + '*Manual.bsq'
    else:
        if( acqRef == 3 ):
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170829\HYMSY\End Products_Manual'    # Hyper - Acq 3
            os.chdir( HomeDir )
            PatternUpload = '.\\' + '*Manual.bsq'
        else:
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20171026\HYMSY\F1'   # F1
            os.chdir( HomeDir )
            PatternUpload = '.\\' + '*Manual.bsq'
else:
    if (acqRef == 1):
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20140515\End Products manual_update'  # Hyper - Acq 1
        os.chdir( HomeDir )
        PatternUpload = '.\\' + '*Manual.bsq'
    else:
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20141014\End Products\Update_manual_20190314'  # Hyper - Acq 3  (calib02)
        os.chdir( HomeDir )
        PatternUpload = '.\\' + '*Manual.bsq'
HypFilesAll = glob.glob(PatternUpload)
os.chdir( HomeDir )

### Flight lines with comparable radiometric quality

Keeps only the flight lines judged comparable in radiometric quality. *(Original cell 3.)*

In [ ]:
if yearRef == 2017:
    if( acqRef == 1 ):
        SubRef = [0,1]        # acq 1
    else:
        if( acqRef == 3 ):
            SubRef = [0,1,2]  # acq 3
        else:
            SubRef = [0,1]
else:
    if( acqRef == 1 ):
        SubRef = [0,1,2]     # acq 1
    else:
        SubRef = [0,1,2,3]   # acq 2  ;  remove image 2 (which is kept here just to derive an aligned version of the image that will be removed...
# filter made in 2 steps, by order - using this reference - and by name using flightline a little further below in the script)

ImgData = []
for ii in range( len(SubRef) ):
    ImgData.append( HypFilesAll[ SubRef[ii] ] )  # use the 4th flight line for the moment | better coverage ...

# using gdal
NoData_value  = -9999
img_float_all = list()
geotransform  = list()
projection    = list()
for ii in range( len(ImgData) ): # images
    img  = gdal.Open( ImgData[ii], gdal.GA_ReadOnly )
    geotransform.append( img.GetGeoTransform() )
    projection.append(   img.GetProjection() )
    imgR = img.GetRasterBand(1)
    imgA = img.ReadAsArray().astype( np.int16 )
    imgR = None
    img  = None
    imgAR  = np.reshape( imgA, (imgA.shape[1], imgA.shape[2], imgA.shape[0]), order='F' )  # , order='F'
    imgAR  = np.swapaxes(imgA,  0, 2)
    imgAR2 = np.swapaxes(imgAR, 0, 1)
    img_float_all.append( imgAR2 )   # keep as array

### Flight lines and plot boundaries

Relates each flight line to the plot boundaries. *(Original cell 5.)*

In [ ]:
# .shp files with GCPs
shpfolder = r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Plots'
os.chdir(shpfolder)
if( yearRef == 2017 ):
    ShpFilesGCP = glob.glob('*FlightLineAdj_'      +  str(acqRef)  + '.shp')[0]  # folder with .shp indicating reference tiles
else:
    ShpFilesGCP = glob.glob('*FlightLineAdj_2014_' +  str(acqRef)  + '.shp')[0]  # folder with .shp indicating reference tiles
GCP_p = gpd.read_file( ShpFilesGCP )

# retain only flight lines considered..
GCP_p = GCP_p[ GCP_p.FlighLine.isin( list(np.array(SubRef)+1) ) ].copy(deep=True)

pixelSizeX_all = []
pixelSizeY_all = []
x_min_all = []
y_max_all = []
x_max_all = []
y_min_all = []
x_res_all = []
y_res_all = []
NoData_value = -9999
for ii in range(len(img_float_all)):
    pixelSizeX_all.append( round(geotransform[ii][1], 3) )
    pixelSizeY_all.append( round(geotransform[ii][5], 3) )
    x_min_all.append( geotransform[ii][0] )
    y_max_all.append( geotransform[ii][3] )
    x_max_all.append( x_min_all[ii] + pixelSizeX_all[ii] * img_float_all[ii].shape[0] )   # add distance using pixel size
    y_min_all.append( y_max_all[ii] + pixelSizeY_all[ii] * img_float_all[ii].shape[1] )
    x_res_all.append( int( round((x_max_all[ii] - x_min_all[ii]) /  pixelSizeX_all[ii]) ) )
    y_res_all.append( int( round((y_max_all[ii] - y_min_all[ii]) / (pixelSizeY_all[ii]*-1)) ) )

# gdal
ProjectionD = copy.deepcopy( projection )
TransformD  = copy.deepcopy( geotransform )

# get deviation of tiles locations from GCPs true positions
FLRef = sorted( GCP_p.loc[:,'FlighLine'].unique() )
xdist = []
ydist = []
TransformNAff_all = []
for ii in range( len(FLRef) ):
    xdist.append( GCP_p.loc[ (GCP_p.loc[:,'FlighLine'] == FLRef[ii]) & (GCP_p.loc[:,'PointRef'] == 'New'), : ].geometry.x.tolist()[0] - \
                  GCP_p.loc[ (GCP_p.loc[:,'FlighLine'] == FLRef[ii]) & (GCP_p.loc[:,'PointRef'] == 'Old'), : ].geometry.x.tolist()[0] )
    ydist.append( GCP_p.loc[ (GCP_p.loc[:,'FlighLine'] == FLRef[ii]) & (GCP_p.loc[:,'PointRef'] == 'New'), : ].geometry.y.tolist()[0] - \
                  GCP_p.loc[ (GCP_p.loc[:,'FlighLine'] == FLRef[ii]) & (GCP_p.loc[:,'PointRef'] == 'Old'), : ].geometry.y.tolist()[0] )
    TransformNAff_all.append( [ x_min_all[ii] - xdist[ii], pixelSizeX_all[ii], 0, y_max_all[ii] - ydist[ii], 0, pixelSizeY_all[ii] ] )

TransformN = copy.deepcopy( geotransform )

if( yearRef == 2017 ):
    if( acqRef == 1 ):
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170509\HYMSY'                          # Hyper - Acq 1
    else:
        if( acqRef == 3 ):
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170829\HYMSY\End Products_Manual'  # Hyper - acq 3
        else:
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20171026\HYMSY\F1'
else:
    if( acqRef == 1 ):
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20140515\End Products manual_update'           # Hyper - Acq 1
    else:
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20141014\End Products\Update_manual_20190314'  # Hyper - acq 2  (calib02)
os.chdir( HomeDir )

# band ref - reference for number of bands
in_data      = gdal.Open(ImgData[0], gdal.GA_ReadOnly)
in_data_Meta = in_data.GetMetadata()
BandsRef     = list( in_data_Meta )
in_data      = None
in_data_Meta = None

BandRefO = []
for elmnt in range(len(BandsRef)):
    if ('Band' in BandsRef[elmnt]):
        BandRefO.append(elmnt)
    else:
        continue

## 3. Spectra of each plot

### Adjusted images and plot coordinates

Reads the adjusted flight lines of the acquisition, their orientation files and the plot coordinates. *(Original cell 7.)*

In [ ]:
# get reference to Flght Lines after positionning correction...
if yearRef == 2017:
    if( acqRef == 1 ):
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170509\HYMSY'                          # Hyper - Acq 1
    else:
        if( acqRef == 3 ):
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170829\HYMSY\End Products_Manual'  # Hyper - acq 3
        else:
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20171026\HYMSY\F1'
else:
    if (acqRef == 1):
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20140515\End Products manual_update'           # Hyper - Acq 1
    else:
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20141014\End Products\Update_manual_20190314'  # Hyper - Acq 2
os.chdir( HomeDir )
PatternUpload = '.\\' + '*Manual_A.bsq'
ImgData = glob.glob(PatternUpload)

# retain only images with good radiometric quality - new
ImgData           = [ ImgData[val] for val in range(len(ImgData)) if val in SubRef ]
# not necessary for 2017 since 'extra' info has already been removed..
# if( all([ (yearRef == 2014), (acqRef == 2) ]) | all([ (yearRef == 2017), (acqRef == 4) ]) ):   # for acq 2 of 2014 use only 3 flight lines since nbr 3 was taken at higher altitude
if( all([ (yearRef == 2014), (acqRef == 2) ]) ):   # for acq 2 of 2014 use only 3 flight lines since nbr 3 was taken at higher altitude
    ImgDataT       = list()
    img_float_allT = list()
    geotransformT  = list()
    projectionT    = list()
    for ii in range( len(ImgData) ):  # flight lines
        if 'FlightLine03' in ImgData[ii]:
            continue
        else:
            ImgDataT.append( ImgData[ii] )
            img_float_allT.append( img_float_all[ii] )
            geotransformT.append( geotransform[ii] )
            projectionT.append(   projection[ii] )
    ImgData       = copy.deepcopy( ImgDataT )
    img_float_all = copy.deepcopy( img_float_allT )
    geotransform  = copy.deepcopy( geotransformT )
    projection    = copy.deepcopy( projectionT )

# find nearest flight line to each plot 'row' center base on HYMSY orientation file
shpfolder = r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Plots'
os.chdir(shpfolder)
if( dataTypeSel == 'Full' ):
    # get plot coordinates
    PatternUpload = '*Velden_UTM_WGS84_mod_v3.shp'  # plots
    ShpFilesAll = glob.glob(PatternUpload)
    with fiona.open( ShpFilesAll[0], "r" ) as shapefile:  # change number according to shape file to plot....
        features   = [ feature["geometry"] for feature in shapefile ]
        featuresID = [ feature["properties"]["Id"] for feature in shapefile ]
    Plot_coords = gpd.read_file( ShpFilesAll[0] )
else:
    # buffer with 2*pixel size...
    PatternUpload = '*plot_boundBuff.shp'
    ShpFilesAll = glob.glob(PatternUpload)
    with fiona.open(ShpFilesAll[0], "r") as shapefile:
        features = [feature["geometry"] for feature in shapefile]
        featuresID = [feature["properties"]["Id"] for feature in shapefile]
    Plot_coords = gpd.read_file( ShpFilesAll[0] )

# sort according to plot ID
PlotNbrRef   = sorted(Plot_coords.Id.tolist())
featuresSort = []
for ii in range( len(PlotNbrRef) ):  # plot Id
    locT = [ featuresID[val] == PlotNbrRef[ii] for val in range(len(featuresID)) ]
    locT = [ val for val, val2 in enumerate(locT) if val2 ]
    featuresSort.append( features[locT[0]] )

# get flight line for comparison with plot location - external HD
if( yearRef == 2017 ):
    if( acqRef == 1 ):
        HomeDir = r'<RAW_DATA>\2017\20170509_Haus Riswick\HYMSY\Processing'  # Hyper - Acq 1
    else:
        if( acqRef == 3 ):
            HomeDir = r'<RAW_DATA>\2017\20170829-riswick\HYMSY\Processing'   # Hyper - acq 3
        else:
            HomeDir = r'<RAW_DATA>\2017\20171026_haus_riswick\hymsy_flight1\Processing'  # Hyper - acq 4
else:
    if( acqRef == 1 ):
        HomeDir = r'<RAW_DATA>\2014-2012\2014-05-15 Haus Riswick\Flight 1 - HYMSY\Processing'  # Hyper - Acq 1
    else:
        HomeDir = r'<RAW_DATA>\2014-2012\2014-10-14 Haus Riswick\Processing'                   # Hyper - Acq 2
os.chdir( HomeDir )
PatternUpload = '*Orientations.txt'
OrienFiles    = glob.glob(PatternUpload)

# retain only images with good radiometric quality - new
OrienFiles = [ OrienFiles[val] for val in range(len(OrienFiles)) if val in SubRef ]
# if( all([ (yearRef == 2014), (acqRef == 2) ]) | all([ (yearRef == 2017), (acqRef == 4) ]) ):   # for acq 2 of 2014 and acq 4 of 2017 use only 3 flight lines since nbr 3 was taken with higher
if( all([ (yearRef == 2014), (acqRef == 2) ]) ):
    OrienFilesT = []
    for ii in range(len(OrienFiles)):  # flight lines
        if 'FlightLine03' in OrienFiles[ii]:
            continue
        else:
            OrienFilesT.append( OrienFiles[ii] )
    OrienFiles = copy.deepcopy( OrienFilesT )
OrienFiles_df = []
for ii in range( len(OrienFiles) ):  # flight line
    OrienFiles_df.append( pd.read_csv( OrienFiles[ii], sep="\t", header=0, encoding="ISO-8859-1") )   # , encoding='ascii'   , encoding="ISO-8859-1" , header=None

# set homeDir
if( yearRef == 2017 ):
    if( acqRef == 1 ):
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170509\HYMSY'                      # Hyper - Acq 1
    else:
        if( acqRef == 3 ):
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170829\HYMSY\End Products_Manual'  # Hyper - acq 3
        else:
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20171026\HYMSY\F1'
else:
    if( acqRef == 1 ):
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20140515\End Products manual_update'           # Hyper - Acq 1
    else:
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20141014\End Products\Update_manual_20190314'  # Hyper - Acq 2
os.chdir( HomeDir )

### Choice of flight line for each plot

Counts the pixels of each plot in each flight line and its distance to the line, to choose the flight line used for each plot. *(Original cell 8.)*

In [ ]:
pxlCount = []
distFL   = []
areaCov  = []
plt.close('all')
for ii in range( len(PlotNbrRef) ):  # plot
# for ii in range( 20 ):  # plot - run manually
    PlotSel = Plot_coords.loc[ Plot_coords.Id == PlotNbrRef[ii],: ]
    PlotSelX = [ PlotSel['geometry'].exterior.iloc[0,].coords.xy[0][val] for val in range(len(PlotSel['geometry'].exterior.iloc[0,].coords.xy[0])) ]
    PlotSelY = [ PlotSel['geometry'].exterior.iloc[0,].coords.xy[1][val] for val in range(len(PlotSel['geometry'].exterior.iloc[0,].coords.xy[1])) ]
    if( ii == 0 ):
        fig = plt.figure(1, figsize=(8,8), dpi=150)
        iplot = 0
    pxlCountT      = []
    distFLT        = []
    coordSUSelPolT = []
    for jj in range( len(ImgData) ):  # flight line
        out_image, out_transform = rasterio.mask.mask( rasterio.open(ImgData[jj]), [ featuresSort[ii] ],
                                                       filled=False, crop=True, all_touched=False )
        out_imageA = out_image.astype('float').filled(fill_value=-9999)

        # get pixels rows | columns and their coordinates - only pixels without NoData... equivalent to all() in pandas
        xyRef_all = []
        for elmnt in range( out_imageA.shape[0] ):  # bands
            ySU, xSU = np.where( np.logical_and( (out_imageA[elmnt,:,:] != -9999),  (out_imageA[elmnt,:,:] != 0)) )   # selecting only pixels != -9999 and 0 in all bands
            PxlRef = [ str(ySU[val]) + '_' + str(xSU[val]) for val in range( len(ySU) ) ]  # reference for pixel
            xyRef_all.append( [ (PxlRef[val], ySU[val], xSU[val]) for val in range(len(ySU)) ] )
        for elmnt in range( len(xyRef_all)-1 ): # bands
            if( elmnt == 0 ):
                a = {t[0]:t[1:] for t in xyRef_all[0]}       # pxls with data first band
                b = {t[0]:t[1:] for t in xyRef_all[elmnt+1]} # pxls with data another band
                common_keys = list( set(a.keys())&set(b.keys()) )
                a = {k: a[k] for k in common_keys}
            else:
                b = {t[0]:t[1:] for t in xyRef_all[elmnt+1]} # pxls with data another band
                common_keys = list(set(a.keys())&set(b.keys()))
                a = {k: a[k] for k in common_keys}

            keysRef = list( a.keys() )
            ySU = [ a[ keysRef[val] ][0] for val in range(len(keysRef)) ]
            xSU = [ a[ keysRef[val] ][1] for val in range(len(keysRef)) ]
            xySU    = [ ( xSU[val], ySU[val] ) for val in range(len(xSU))  ]
            coordSU = [ out_transform * Affine.translation(0.5, 0.5) * xySU[val] for val in range(len(xySU)) ]

        # using cumulative mask .. remove all NoData and zer...
        pxlCountT.append( len(coordSU) )  # pixels counting...

        if( jj == 0 ):
            iplot += 1
            ax = fig.add_subplot(5,10,iplot)
            ax.imshow( np.matrix( out_imageA[40,:,:] ), cmap=plt.get_cmap('gray') )  # even if without data will show since contrats between -9999 and 0 is large...
            plt.axis('off')
            plt.xticks([])
            plt.yticks([])

        # distance
        PlotSelC = Point( PlotSel.centroid.geometry.x, PlotSel.centroid.geometry.y )
        FL_point = [ Point( OrienFiles_df[jj].loc[:,'X/Easting'].tolist()[val], OrienFiles_df[jj].loc[:,'Y/Northing'].tolist()[val] ) for val in range(OrienFiles_df[jj].shape[0]) ]
        FL_line  = LineS( FL_point )
        distFLT.append( PlotSelC.distance(FL_line) )

        # based on area dscribed by nbr of pxls - easier approach
        if( len(coordSU) >= 1 ):  # at least one pxl
            coordSUSelPolT.append( geotransform[jj][1]*geotransform[jj][1]*len(coordSU) )
        else:
            coordSUSelPolT.append( 0 )

    pxlCount.append( pxlCountT )
    distFL.append( distFLT )
    areaCov.append( coordSUSelPolT )

# visual quality check... G == good ; B == bad - new version | only radiometrically comparable flight Lines
if( yearRef == 2017 ):
    if( acqRef == 1 ):
        QualAll = ['B', 'G']           # 2017 - acq 1
    else:
        if( acqRef == 3 ):
            QualAll = ['G', 'G', 'G']  # 2017 - acq 3
        else:
            QualAll = ['G', 'G']       # 2017 - acq 4 - New
else:
    if( acqRef == 1 ):
        QualAll = ['G', 'G', 'G']  # 2014 - acq 1
    else:
        QualAll = ['G', 'G', 'B']  # 2014 - acq 2

# using area coverage instead of number of pixels
MinThreVal = 0.10
FlSel      = []
for ii in range( len(pxlCount) ):  # plot
    QualAllS  = [ val for val in range(len(QualAll)) if QualAll[val] == 'G' ]  # position of flight lines with good quality
    distFLS   = [ distFL[ii][ val ] for val in QualAllS ]                      # distance of flight lines with good quality
    areaCovS  = [ areaCov[ii][ val ] for val in QualAllS ]                     # nbr of pixels of flight lines with good quality
    QualAllSb = [ val for val in range(len(QualAll)) if QualAll[val] == 'B' ]  # same as above but for bad quality fligth lines
    distFLSb  = [ distFL[ii][ val ] for val in QualAllSb ]
    areaCovSb = [ areaCov[ii][ val ] for val in QualAllSb ]
    if( any( list(x >= (max(areaCov[ii])-(MinThreVal*(max(areaCov[ii])))) for x in areaCovS) + \
             [ (np.nanmax(areaCov[ii]) in areaCovS) ] ) ):  # if any flight line with good quality fullfil the requirements of minimum area
        QualAllSThrs = [ QualAllS[val] for val in range(len(QualAllS)) if areaCovS[val] >= (max(areaCov[ii])-(MinThreVal*(max(areaCov[ii])))) ]  # flight lines with more pixels than threshold
        if( (len(QualAllSThrs) == 0) ):
            FlSel.append( QualAllS[ np.where(np.array(areaCovS) == np.nanmax(areaCov[ii]))[0][0] ] )  # largest coverage
        else:
            distFLSThrs  = [ distFL[ii][val] for val in QualAllSThrs ]            # distance of flight lines with good quality
            FlSel.append( QualAllSThrs[ distFLSThrs.index(min(distFLSThrs)) ]  )  # select the closets flight line that fullfil the requirements of minimum area
    else:
        QualAllSThrsb = [ QualAllSb[val] for val in range(len(QualAllSb)) if areaCovSb[val] >= (max(areaCov[ii])-(MinThreVal*(max(areaCov[ii])))) ]  # bad flight lines with more pixels than threshold
        if( (len(QualAllSThrsb) == 0) ):
            FlSel.append( QualAllSb[ np.where(np.array(areaCovSb) == np.nanmax(areaCov[ii]))[0][0] ] )  # largest coverage
        else:
            distFLSThrsb  = [ distFL[ii][val] for val in QualAllSThrsb ]             # distance of flight lines with bad quality
            FlSel.append( QualAllSThrsb[ distFLSThrsb.index(min(distFLSThrsb)) ]  )  # select the closets bad flight line that fullfil the requirements of minimum nbr of pixels

shpfolder = r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Plots'
os.chdir(shpfolder)
PatternUpload = '*Velden_UTM_WGS84_mod_v3.shp'  # plots
ShpFilesAll = glob.glob(PatternUpload)
with fiona.open(ShpFilesAll[0], "r") as shapefile:  # change number according to shape file to plot....
    featuresT = [feature["geometry"] for feature in shapefile]
def plot_line(ax, ob):
    x, y = ob.xy
    ax.plot(x, y, color='red', alpha=0.7, linewidth=3, solid_capstyle='round', zorder=2)

crs = ccrs.epsg(32632)
for ii in range( len(img_float_all) ):  # flight line...
    image = ii+1  # 1-3 | one by one manually
    xmin = geotransform[(image-1)][0]
    xmax = geotransform[(image-1)][0] + (geotransform[(image-1)][1] * img_float_all[(image-1)].shape[1])
    ymin = geotransform[(image-1)][3] + (geotransform[(image-1)][5] * img_float_all[(image-1)].shape[0])
    ymax = geotransform[(image-1)][3]
    band = 41
    fig = plt.figure(10+ii, figsize=(3,3), dpi=150)
    ax  = plt.axes(projection=crs)
    ax.imshow( img_float_all[(image-1)][:,:,band], cmap='gray',
                origin='upper', extent=(xmin, xmax, ymin, ymax),
                transform=crs, interpolation='nearest',
                vmin=np.percentile(img_float_all[(image-1)][:,:,band], 0.25),
                vmax=np.percentile(img_float_all[(image-1)][:,:,band], 99.25) )

    # plot plots
    patches = [PolygonPatch(feature, edgecolor="b", facecolor="none", linewidth=2) for feature in featuresT]
    ax.add_collection( matplotlib.collections.PatchCollection(patches, match_original=True) )
    FL_point = [ Point( OrienFiles_df[ii].loc[:,'X/Easting'].tolist()[val], OrienFiles_df[ii].loc[:,'Y/Northing'].tolist()[val] ) for val in range(OrienFiles_df[ii].shape[0]) ]
    FL_line  = LineS( FL_point )
    plot_line(ax=ax, ob=FL_line)

### Extraction of spectra

Extracts the spectra of each sampling unit, with and without the buffer area, and saves them. *(Original cell 9.)*

In [ ]:
shpfolder = r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Plots'
os.chdir( shpfolder )
if( dataTypeSel == 'Full' ):
    # full extent plots
    PatternUpload = '*Velden_UTM_WGS84_mod_v3.shp'
    ShpFilesAll   = glob.glob(PatternUpload)
    with fiona.open(ShpFilesAll[0], "r") as shapefile:
        features   = [ feature["geometry"] for feature in shapefile ]
        featuresID = [ feature["properties"]["Id"] for feature in shapefile ]
    plot_bound = gpd.read_file( ShpFilesAll[0] )
    plot_bound.columns
else:
    # buffer with 2*pixel size
    PatternUpload = '*plot_boundBuff.shp'
    ShpFilesAll   = glob.glob(PatternUpload)
    with fiona.open(ShpFilesAll[0], "r") as shapefile:
        features   = [ feature["geometry"] for feature in shapefile ]
        featuresID = [ feature["properties"]["Id"] for feature in shapefile ]
    plot_bound = gpd.read_file(ShpFilesAll[0])
    plot_bound.columns

# subset plots by exp row
featuresSub   = []
plot_boundSub = []
BlocRef       = [1,2,3]
for ii in range( len(BlocRef) ):
    BlocRefTemp = (plot_bound.Block == BlocRef[ii]).tolist()
    BlocRefTempN = [ val for val in range(len(BlocRefTemp)) if BlocRefTemp[val] == True ]
    featuresSub.append( [ features[ val ] for val in BlocRefTempN ] )
    print( len(featuresSub[ii]) )
    plot_boundSub.append( plot_bound[ plot_bound['Block'] == BlocRef[ii] ] )

# set HomeDir
if (yearRef == 2017):
    if (acqRef == 1):
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170509\HYMSY'                          # Hyper - Acq 1
    else:
        if( acqRef == 3 ):
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170829\HYMSY\End Products_Manual'  # Hyper - Acq 3
        else:
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20171026\HYMSY\F1'                   # F1
else:
    if( acqRef == 1 ):
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20140515\End Products manual_update'           # Hyper - Acq 1
    else:
        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20141014\End Products\Update_manual_20190314'  # Hyper - Acq 2
os.chdir( HomeDir )

# extraction v2 - using nearest Flight line (considering flight line quality and coverage) to the plot center...

# order plots by Id
plotIdRef = sorted( plot_bound.Id.tolist() )
for ii in pbar.progressbar( range(len(plotIdRef)) ):  # plot...
    if( ii == 0 ):
        plot_boundId = plot_bound.loc[ (plot_bound.Id == plotIdRef[ii]),: ].copy(deep=True)
        plot_boundId = plot_boundId.reset_index( drop=True )
    else:
        plot_boundId = plot_boundId.append( plot_bound.loc[ plot_bound.Id == plotIdRef[ii],: ],  ignore_index=True )

# sort Fiona according to plot ID
featuresSort = []
for ii in range( len(plotIdRef) ):  # plot Id
    locT = [ featuresID[val] == plotIdRef[ii] for val in range(len(featuresID)) ]
    locT = [val for val, val2 in enumerate(locT) if val2]
    featuresSort.append( features[locT[0]] )

NAValue = 0

plt.close("all")
for ii in range( len(FlSel) ): # experimental plot
    if( ii == 0 ):
        fig = plt.figure(1, figsize=(8,8), dpi=150)
        iplot = 0
    out_image, out_transform = rasterio.mask.mask( rasterio.open(ImgData[ FlSel[ii] ]), [ featuresSort[ii] ],
                                                   filled=False, crop=True, all_touched=False )
    out_imageA = out_image.filled(fill_value=NAValue)
    iplot += 1
    ax = fig.add_subplot(5,10,iplot)
    ax.imshow( np.matrix( out_imageA[40,:,:] ), cmap=plt.get_cmap('gray') )
    plt.axis('off')
    plt.xticks([])
    plt.yticks([])

    # transform image
    bands, rows, cols = np.shape( out_imageA )
    out_imageAS       = np.rollaxis(out_imageA, 0, 3)
    imageT = np.reshape( out_imageAS, (rows*cols, bands) )

    xyRef_all       = np.full( out_imageA.shape[1:], False )
    xyRef_all_range = np.full( out_imageA.shape[1:], True )
    for elmnt in range( out_imageA.shape[0] ):  # bands
        maskTempT       = (out_imageA[elmnt,:,:] != NAValue)
        maskTempT_range = (out_imageA[elmnt,:,:] <= 10000)
        xyRef_all = xyRef_all | maskTempT
        xyRef_all_range = xyRef_all_range & maskTempT_range  # only keep pixels in which there is no value above 10000 across all bands
    xyRef_all = xyRef_all & xyRef_all_range
    ySU, xSU   = np.where( xyRef_all != False )
    PxlRef     = [ str(ySU[val]) + '_' + str(xSU[val]) for val in range( len(ySU) ) ]  # reference for pixel
    xyRef_allT = [ (PxlRef[val], ySU[val], xSU[val])   for val in range(len(ySU)) ]
    xyRef_all  = [ xyRef_allT ]*out_imageA.shape[0]

    ySU, xSU = rasterio.transform.xy(out_transform, rows=ySU.tolist(), cols=xSU.tolist(), offset='center')
    coordSU  = [ (ySU[val], xSU[val]) for val in range(len(xSU)) ]

    # Meta data
    Iref  = list( itertools.repeat( plot_boundId.iloc[ii,:].Id,        imageT.shape[0]) )  # using
    Nref  = list( itertools.repeat( plot_boundId.iloc[ii,:].Naam,      imageT.shape[0]) )
    Tref  = list( itertools.repeat( plot_boundId.iloc[ii,:].Treat,     imageT.shape[0]) )
    Bref  = list( itertools.repeat( plot_boundId.iloc[ii,:].Block,     imageT.shape[0]) )
    Rref  = list( itertools.repeat( plot_boundId.iloc[ii,:].Replicate, imageT.shape[0]) )
    TCref = list( itertools.repeat( plot_boundId.iloc[ii,:].TreatCode, imageT.shape[0]) )

    # get dataframe
    col_ref = ['Id', 'Naam', 'Treat', 'Block', 'Replicate', 'TreatCode'] + [ 'b'+ [str(x) if x>=100 else ( '00' + str(x) if x<10 else '0'+str(x) )][0] for x in range(0, 101,1) ]

    if( all([ (ii == 0) ]) ):
        for val in range( imageT.T.shape[0] ):  # bands
            if( val == 0 ):
                dict1 = { val : imageT.T[val,:] }
            else:
                dict1[val] = imageT.T[val,:]
        dict2 = { 'b'+ [str(x) if x>=100 else ( '00' + str(x) if x<10 else '0'+str(x) )][0]: dict1[x] for x in dict1.keys() }
        dict2['Id']        = Iref
        dict2['Naam']      = Nref
        dict2['Treat']     = Tref
        dict2['Block']     = Bref
        dict2['Replicate'] = Rref
        dict2['TreatCode'] = TCref
        imageT_df = pd.DataFrame( dict2, index=range(imageT.shape[0]), columns=col_ref )
        BandRef   = [ 'b'+ [str(x) if x>=100 else ( '00' + str(x) if x<10 else '0'+str(x) )][0] for x in range(0, 101,1) ]
        maskT     = np.full(out_imageA.shape[1:], False)
        maskT[ [ xyRef_all[0][val][1] for val in range(len(xyRef_all[0])) ], \
               [ xyRef_all[0][val][2] for val in range(len(xyRef_all[0])) ] ] = True
        maskT     = np.reshape( maskT, (rows*cols, 1) )
        imageT_df = imageT_df[ maskT ].copy(deep=True)

        geometryAll = [ Point(val) for val in coordSU ]  # get geodatframe
        crs_ref     = 'epsg:32632'
        imageT_gdf  = gpd.GeoDataFrame( imageT_df, crs=crs_ref, geometry=geometryAll )
    else:
        for val in range( imageT.T.shape[0] ):  # bands
            if( val == 0 ):
                dict1 = { val : imageT.T[val,:] }
            else:
                dict1[val] = imageT.T[val,:]
        dict2 = { 'b'+ [str(x) if x>=100 else ( '00' + str(x) if x<10 else '0'+str(x) )][0]: dict1[x] for x in dict1.keys() }
        dict2['Id']        = Iref
        dict2['Naam']      = Nref
        dict2['Treat']     = Tref
        dict2['Block']     = Bref
        dict2['Replicate'] = Rref
        dict2['TreatCode'] = TCref
        imageT_df_temp = pd.DataFrame( dict2, index=range(imageT.shape[0]), columns=col_ref )
        maskT = np.full(out_imageA.shape[1:], False)
        maskT[ [ xyRef_all[0][val][1] for val in range(len(xyRef_all[0])) ], \
               [ xyRef_all[0][val][2] for val in range(len(xyRef_all[0])) ] ] = True
        maskT = np.reshape( maskT, (rows*cols, 1) )
        imageT_df_temp = imageT_df_temp[ maskT ].copy(deep=True)
        imageT_df      = imageT_df.append(imageT_df_temp, ignore_index=True)

        geometryAll = [ Point(val) for val in coordSU ]  # get geodatframe
        crs_ref     = 'epsg:32632'
        imageT_gdf_temp = gpd.GeoDataFrame( imageT_df_temp, crs=crs_ref, geometry=geometryAll )
        imageT_gdf      = imageT_gdf.append( imageT_gdf_temp, ignore_index=True )

# plot
imageT_df.shape

if( yearRef == 2017 ):
    datesR = [ '20170509', '20170829', '20171026' ]
else:
    datesR = [ '20140515', '20141014' ]

# save pd dataframe - HYMSY images - select accrding to FL distance, quality and coverage
compRefT = 'New_UpD'

if( yearRef == 2017 ):
    if( dataTypeSel == 'Full' ):
        if( acqRef == 1 ):
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170509\HYMSY' )  # Hyper - Acq 1
            imageT_df.to_pickle( '2017-05-09_spectra_SU_VegProp_FLD_'     + compRefT + '.pkl')
            imageT_gdf.to_pickle('2017-05-09_spectra_SU_geo_VegProp_FLD_' + compRefT + '.pkl')
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
            imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[0] + '_spectra_geo_SU_VegProp_FLD_' + compRefT + '.shp')
        else:
            if( acqRef == 3 ):
                os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170829\HYMSY\End Products_Manual' )  # Hyper - acq 3
                imageT_df.to_pickle( '2017-08-29_spectra_SU_VegProp_FLD_'     + compRefT + '.pkl')
                imageT_gdf.to_pickle('2017-08-29_spectra_SU_geo_VegProp_FLD_' + compRefT + '.pkl')
                os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
                imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[1] + '_spectra_geo_SU_VegProp_FLD_' + compRefT + '.shp')
            else:
                os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20171026\HYMSY\F1' )  # Hyper - acq 4
                imageT_df.to_pickle( '2017-10-26_spectra_SU_VegProp_FLD_'     + compRefT + '.pkl')
                imageT_gdf.to_pickle('2017-10-26_spectra_SU_geo_VegProp_FLD_' + compRefT + '.pkl')
                os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
                imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[2] + '_spectra_geo_SU_VegProp_FLD_' + compRefT + '.shp')
    else:
        if( acqRef == 1 ):
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170509\HYMSY' )  # Hyper - Acq 1
            imageT_df.to_pickle( '2017-05-09_spectra_SU_VegProp_B_FLD_'     + compRefT + '.pkl')
            imageT_gdf.to_pickle('2017-05-09_spectra_SU_geo_VegProp_B_FLD_' + compRefT + '.pkl')
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
            imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[0] + '_spectra_geo_SU_VegProp_B_FLD_' + compRefT + '.shp')
        else:
            if acqRef == 3:
                os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170829\HYMSY\End Products_Manual' )  # Hyper - acq 3
                imageT_df.to_pickle( '2017-08-29_spectra_SU_VegProp_B_FLD_'     + compRefT + '.pkl')
                imageT_gdf.to_pickle('2017-08-29_spectra_SU_geo_VegProp_B_FLD_' + compRefT + '.pkl')
                os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
                imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[1] + '_spectra_geo_SU_VegProp_B_FLD_' + compRefT + '.shp')
            else:
                os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20171026\HYMSY\F1' )  # Hyper - acq 4
                imageT_df.to_pickle( '2017-10-26_spectra_SU_VegProp_B_FLD_'     + compRefT + '.pkl')
                imageT_gdf.to_pickle('2017-10-26_spectra_SU_geo_VegProp_B_FLD_' + compRefT + '.pkl')
                os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
                imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[2] + '_spectra_geo_SU_VegProp_B_FLD_' + compRefT + '.shp')
else:
    if( dataTypeSel == 'Full' ):
        if( acqRef == 1 ):
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20140515\End Products manual_update' )  # Hyper - Acq 1
            imageT_df.to_pickle( '2014-05-15_spectra_SU_VegProp_FLD_'     + compRefT + '.pkl')
            imageT_gdf.to_pickle('2014-05-15_spectra_SU_geo_VegProp_FLD_' + compRefT + '.pkl')
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
            imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[0] + '_spectra_geo_SU_VegProp_FLD_' + compRefT + '.shp')
        else:
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20141014\End Products\Update_manual_20190314' )  # Hyper - acq 2
            imageT_df.to_pickle( '2014-10-14_spectra_SU_VegProp_FLD_'     + compRefT + '.pkl')
            imageT_gdf.to_pickle('2014-10-14_spectra_SU_geo_VegProp_FLD_' + compRefT + '.pkl')
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
            imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[1] + '_spectra_geo_SU_VegProp_FLD_' + compRefT + '.shp')
    else:
        if( acqRef == 1 ):
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20140515\End Products manual_update' )  # Hyper - Acq 1
            imageT_df.to_pickle( '2014-05-15_spectra_SU_VegProp_B_FLD_'     + compRefT + '.pkl')
            imageT_gdf.to_pickle('2014-05-15_spectra_SU_geo_VegProp_B_FLD_' + compRefT + '.pkl')
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
            imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[0] + '_spectra_geo_SU_VegProp_B_FLD_' + compRefT + '.shp')
        else:
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20141014\End Products\Update_manual_20190314' )  # Hyper - acq 2
            imageT_df.to_pickle( '2014-10-14_spectra_SU_VegProp_B_FLD_'     + compRefT + '.pkl')
            imageT_gdf.to_pickle('2014-10-14_spectra_SU_geo_VegProp_B_FLD_' + compRefT + '.pkl')
            os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )
            imageT_gdf.to_file(driver = 'ESRI Shapefile', filename= datesR[1] + '_spectra_geo_SU_VegProp_B_FLD_' + compRefT + '.shp')

## 4. Shared data

### Files for the published dataset

Writes the vector files with the spectra and plot boundaries shared in the dataset. *(Original cell 11.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Extrated_spectra' )

datesR = [ '20140515', '20141014', '20170509', '20170829', '20171026' ]

compRefT = 'New_UpD'

saveOpt = False

# Extracted spectra for buffer areas
for ii in pbar.progressbar( range( len(datesR) ) ):

    InFileT   = gpd.read_file( datesR[ii] + '_spectra_geo_SU_VegProp_B_FLD_' + compRefT + '.shp' )
    TreatRefT = [ InFileT.TreatCode.tolist()[val].split('.')[1] for val in range(InFileT.shape[0]) ]
    bandsRefT = InFileT.columns[6:-1].tolist()
    InFileT   = InFileT[ ['Id', 'Replicate'] + bandsRefT + ['geometry'] ]
    InFileT['Treat'] = TreatRefT
    InFileT = InFileT[ ['Id', 'Treat', 'Replicate'] + bandsRefT + ['geometry'] ]
    if( saveOpt == True ):
        InFileT.to_file(driver = 'ESRI Shapefile', filename= datesR[ii] + '_Spectra_Buffer_UTM_32N' + '.shp')

# Plot boundaries
shpfolder = r'<PROJECT_ROOT>\DataProcessing\2_Data\Vector\Plots'
os.chdir( shpfolder )

# Plot broundries - Full extent
PatternUpload = '*Velden_UTM_WGS84_mod_v3.shp'
InFileT = glob.glob(PatternUpload)
InFileT = gpd.read_file( InFileT[0] )

TreatRefT = [ InFileT.TreatCode.tolist()[val].split('.')[1] for val in range(InFileT.shape[0]) ]
InFileT = InFileT[ ['Id', 'Replicate', 'geometry'] ]
InFileT[ 'Treat' ] = TreatRefT

InFileT = InFileT[ ['Id', 'Treat', 'Replicate', 'geometry'] ]
InFileT = InFileT.sort_values( by='Id', ascending=True )
InFileT = InFileT.reset_index(drop=True)

# save
if( saveOpt == True ):
    InFileT.to_file(driver = 'ESRI Shapefile', filename= 'Plot_Boundaries_UTM_32N' + '.shp')

# Plot broundries - Buffer
PatternUpload = '*plot_boundBuff.shp'
InFileT = glob.glob(PatternUpload)
InFileT = gpd.read_file( InFileT[0] )

TreatRefT = [ InFileT.TreatCode.tolist()[val].split('.')[1] for val in range(InFileT.shape[0]) ]
InFileT = InFileT[ ['Id', 'Replicate', 'geometry'] ]
InFileT[ 'Treat' ] = TreatRefT

InFileT = InFileT[ ['Id', 'Treat', 'Replicate', 'geometry'] ]
InFileT = InFileT.sort_values( by='Id', ascending=True )
InFileT = InFileT.reset_index(drop=True)

# save
if( saveOpt == True ):
    InFileT.to_file(driver = 'ESRI Shapefile', filename= 'Plot_Boundaries_Buffer_UTM_32N' + '.shp')